# Commercial auto claim risk

Estimate the probability of at least one claim during a policy period for each APD or AL policy line. The task provides labelled 2020-2022 history and unlabelled 2023-2024 scoring records. We validate on historical snapshot cohorts, then train the scoring model on all usable labelled history.

## Load the supplied data

Keep this notebook and `risk_scoring.py` in the project directory, with the two CSVs in data/.

In [0]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import StratifiedGroupKFold
from threadpoolctl import threadpool_limits

In [0]:
# Open the notebook from the project folder so data/ and the helper module are found
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data'
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
REPORT_DIR = PROJECT_DIR / 'reports'
sys.path.insert(0, str(PROJECT_DIR))

from risk_scoring import (
    SEED, FEATURES, NUMERIC, CATEGORICAL, DERIVED, UnderwritingFeatures,
    prepare_data, build_model, probability_metrics, training_reference,
    score_policies, save_model, load_model,
)

for directory in [ARTIFACT_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10
})

In [0]:
train_raw = pd.read_csv(DATA_DIR / 'train.csv')
score_raw = pd.read_csv(DATA_DIR / 'score.csv')
print(f'Historical data: {len(train_raw):,} rows, {train_raw.shape[1]} columns')
print(f'Scoring data: {len(score_raw):,} rows, {score_raw.shape[1]} columns')
print('Columns absent from scoring:', sorted(set(train_raw) - set(score_raw)))

In [0]:
train_raw.head()

## Decide what the model may use

Keep predictors that describe the fleet, drivers, prior claims, coverage and underwriting. The model must also accept the same fields in score.csv.

- Target is defined by `claim_count`. Another outcome is `total_loss_amount`.
- Current-period claim dates, payments, status and reporting delays reveal the outcome. Exclude them even if they appear in scoring with placeholders.
- Exclude `late_payment_count`: it accumulates during the current year and is unavailable for an inception-time underwriting score.
- Exclude `has_safety_program`: scoring does not supply it.
- Keep IDs and dates as metadata, outside the predictors. Company IDs support grouped diagnostics. Snapshot dates define the documented file cohorts.

The allowlist contains 18 potential inputs. Premium is known for an issued policy; we test models without premium as a sensitivity comparison.

In [0]:
# FEATURES is a fixed allowlist
excluded_columns = sorted(set(train_raw.columns) - set(FEATURES))
print('Categorical predictors:', ', '.join(CATEGORICAL))
print('Numeric predictors:', ', '.join(NUMERIC))
print('Excluded from X:', ', '.join(excluded_columns))

missing = pd.DataFrame({
    'Historical missing (%)': train_raw[FEATURES].isna().mean() * 100,
    'Scoring missing (%)': score_raw[FEATURES].isna().mean() * 100,
})
missing.loc[missing.gt(0).any(axis=1)].round(2)

Several predictors have missing values. Numeric medians and categorical encoding will be learned inside each model's training pipeline, after splitting the data. Current-claim fields in scoring also contain placeholders such as `9999-01-01` and `-999`. They are excluded fields, not usable claim information.

In [0]:
# Diagnostic only: these current-period fields are never passed to fit()
claim_signals = pd.DataFrame({
    'Current paid amount is positive': train_raw['claim_paid_amount_current_period'].gt(0),
    'First claim report is present': train_raw['first_claim_reported_date'].notna(),
    'Claim status is not No Claim': train_raw['claim_status_current_period'].ne('No Claim'),
})
print('Agreement with the claim-occurrence target (%)')
print(claim_signals.eq(train_raw['claim_count'].gt(0), axis=0).mean().mul(100).round(1))
print('Scoring claim-date placeholders:',
      score_raw['first_claim_reported_date'].dropna().value_counts().to_dict())

Those three historical signals agree with claim occurrence on every row. A model using them could appear excellent while answering a question whose outcome is already known.

## Clean records

Apply the same deterministic rules to training and scoring:
1. Normalize category spelling and case. Remove exact training copies after normalization; conflicting policy IDs raise an error. Scoring IDs must already be unique. Coverage must be APD or AL.
2. Parse inception and expiration dates. Exclude training policies whose expiration is not after inception. A scoring file with an invalid interval fails.
3. Mark invalid numeric inputs unknown: negative amounts/counts, fractional counts, invalid external scores, and deductibles outside the documented values. For the three rows where heavy vehicles exceed total vehicles, mark both counts unknown. Zero fleet sizes are retained unless contradicted by the heavy-vehicle count.

In [0]:
# Fixed validity rules run before splitting; learned medians run after splitting
train, training_quality, rejected_train = prepare_data(train_raw, training=True)
score, scoring_quality, _ = prepare_data(score_raw, training=False)

for name, report in [('Training', training_quality), ('Scoring', scoring_quality)]:
    print(name, {k: v for k, v in report.items() if k != 'missing_predictors'})

rejected_train.to_csv(REPORT_DIR / 'excluded_training_rows.csv', index=False)
rejected_train[['policy_id', 'policy_effective_date', 'policy_expiration_date']].head()

The historical file has 75 duplicate copies after category normalization and 10 reversed coverage intervals. Cleaning leaves 14,990 policy lines. Five negative premiums become unknown, and three conflicting fleet pairs become unknown. Scoring keeps all 3,000 lines.

In [0]:
print('Remaining zero-fleet lines:', train['vehicle_count'].eq(0).sum())
print('Historical policy durations (days):')
print((train['policy_expiration_date'] - train['policy_effective_date']).dt.days.value_counts())
print('Distinct historical insured companies:', train['insured_id'].nunique())
print('Companies appearing in both supplied files:',
      len(set(train['insured_id']) & set(score['insured_id'])))

pd.DataFrame({
    'Historical missing after cleaning': train[NUMERIC].isna().sum(),
    'Scoring missing after cleaning': score[NUMERIC].isna().sum(),
}).loc[lambda frame: frame.gt(0).any(axis=1)]

## Check records

The following checks inspect the raw files and retained records. They verify the file cohorts, row accounting, binary label preservation, outcome enrichment and policy terms.

In [0]:
raw_unique = train_raw.copy()

for column in CATEGORICAL:
    values = raw_unique[column].astype('string').str.strip().replace('', pd.NA)
    raw_unique[column] = (
        values.str.lower() if column == 'business_type'
        else values.str.upper() if column in ['coverage_type', 'state']
        else values.str.title()
    )

raw_unique = raw_unique.drop_duplicates()
retained_source = raw_unique.set_index('policy_id').loc[train['policy_id']]
np.testing.assert_array_equal(train['had_claim'], retained_source['claim_count'].gt(0).astype(int))

assert len(train_raw) == len(train) + training_quality['duplicate_copies_removed'] + len(rejected_train)
assert train['policy_id'].is_unique and score['policy_id'].is_unique
assert set(train['policy_id']).isdisjoint(score['policy_id'])
assert set(train['snapshot_date'].dt.year) == {2020, 2021, 2022}
assert set(score['snapshot_date'].dt.year) == {2023, 2024}
assert (train['policy_expiration_date'] - train['policy_effective_date']).dt.days.eq(365).all()
assert (score['policy_expiration_date'] - score['policy_effective_date']).dt.days.eq(365).all()

reports = pd.to_datetime(train['first_claim_reported_date'], errors='raise')
data_checks = {
    'historical_rows_raw': len(train_raw), 'historical_rows_usable': len(train),
    'duplicate_copies_removed': training_quality['duplicate_copies_removed'],
    'invalid_intervals_excluded': len(rejected_train),
    'scoring_rows_preserved': len(score),
    'claim_positive_rows': int(train['had_claim'].sum()),
    'first_claim_after_snapshot': int(reports.gt(train['snapshot_date']).sum()),
    'scoring_companies_in_history': len(set(train['insured_id']) & set(score['insured_id'])),
    'scoring_companies_total': score['insured_id'].nunique(),
}
print(json.dumps(data_checks, indent=2))

### Capture dates and changes in the scoring population

The file years match `snapshot_date`, not necessarily inception or expiration. A snapshot is a capture event and does not imply a policy has finished. We use it to define offline historical cohorts, following the dictionary's temporal-analysis description.
We also inspect claim report dates. Future claim information in the labelled file demonstrates that these records are retrospectively enriched rather than literal as-of snapshots.

In [0]:
for name, data in [('Historical', train), ('Scoring', score)]:
    snapshots = data['snapshot_date']
    print(f'{name}: snapshots {snapshots.min().date()} to {snapshots.max().date()}; '
          f'inceptions {data["policy_effective_date"].min().date()} '
          f'to {data["policy_effective_date"].max().date()}')
    print('  Captures after expiration:', snapshots.gt(data['policy_expiration_date']).sum())

first_reports = pd.to_datetime(train['first_claim_reported_date'], errors='raise')
print('Claim-positive rows:', train['had_claim'].sum())
print('First claim reported after snapshot:', first_reports.gt(train['snapshot_date']).sum())

category_mix = pd.DataFrame({
    'Historical': train['business_type'].value_counts(normalize=True) * 100,
    'Scoring': score['business_type'].value_counts(normalize=True) * 100,
}).fillna(0)
category_mix.round(2)

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col, title, unit in zip(
    axes,
    ['driver_avg_age', 'prior_year_mileage_000'],
    ['Average driver age', 'Stated fleet mileage'],
    ['Years', 'Thousand miles']
):
    values = pd.concat([train[col], score[col]]).dropna()
    bins = np.histogram_bin_edges(values, bins=22)
    for name, data in [('Historical', train), ('Scoring', score)]:
        observed = data[col].dropna()
        ax.hist(observed, bins=bins, weights=np.full(len(observed), 100 / len(observed)),
                histtype='step', linewidth=2, label=name)
    ax.set(title=title, xlabel=unit, ylabel='Observed policy values (%)')
    ax.legend(frameon=False)
    ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.show()
plt.close()

Scoring drivers average about 39 rather than 42 years of age; average stated mileage rises from about 204,000 to 232,000 fleet miles.

`rideshare` is new in scoring and represents 60 lines, or 2%. Keep these rows and record the new category in the scoring audit. An encoder can accept an unseen category, but that does not establish that its predictions are reliable for a new business segment.

## Define the outcome

Use `had_claim = (claim_count > 0)`, a binary occurrence target permitted by the task. The dictionary describes claims during the policy period. We treat the supplied labels as retrospective period outcomes, rather than declaring that they became available at expiration on a specific historical date.

All retained policy intervals last 365 days. Fleet size and mileage remain predictors. APD and AL can have different risk drivers and baseline rates, so coverage remains an input and both segments receive separate evaluation.

The 14,990 retained lines contain 1,853 claim-positive outcomes (12.36%). All first-claim reports are after their snapshots.

In [0]:
print('Historical claim-positive lines:', train['had_claim'].sum())
print(f'Historical claim occurrence: {train['had_claim'].mean():.2%}')
train.groupby('coverage_type')['had_claim'] \
    .agg(policy_lines='size', claim_positive='sum', observed_claim_rate='mean') \
    .round(4)

## Separate selection, evaluation and final training

Model selection: two expanding windows within 2020-2021. Train on earlier snapshots and validate on the next half-year of 2021. Each pipeline learns its medians, encoding and model from its own fit rows.

Reported evaluation: fit the chosen configuration on 2020-2021 snapshots and evaluate it on 2022 snapshots. This evaluates a later record cohort using supplied outcomes.

Final scoring model: after recording the evaluation, refit the same selected configuration on all 14,990 valid labelled rows. Generate the 3,000 unlabelled scores with this model. Companies can appear in successive cohorts, as 733 of 739 scoring companies also appear in history. A separate group-disjoint diagnostic checks performance without company overlap.

In [0]:
TEST_START = pd.Timestamp('2022-01-01')
fit_mask = train['snapshot_date'].lt(TEST_START)
test_mask = train['snapshot_date'].ge(TEST_START)

fit_data = train.loc[fit_mask].copy()
test_data = train.loc[test_mask].copy()
X_fit, y_fit = fit_data[FEATURES], fit_data['had_claim']
X_test, y_test = test_data[FEATURES], test_data['had_claim']

assert not (fit_mask & test_mask).any()
assert (fit_mask | test_mask).all()
assert set(fit_data.policy_id).isdisjoint(test_data.policy_id)
assert train['snapshot_date'].max() < score['snapshot_date'].min()

print(f'Evaluation fit (2020-2021): {len(fit_data):,} rows')
print(f'Evaluation cohort (2022): {len(test_data):,} rows')
print(f'Final scoring fit, after evaluation: {len(train):,} rows')

In [0]:
validation_windows = [('2021-01-01', '2021-07-01'),
                      ('2021-07-01', '2022-01-01')]
folds, fold_summary = [], []
for start, end in validation_windows:
    earlier = train['snapshot_date'].lt(start)
    later = train['snapshot_date'].ge(start) & train['snapshot_date'].lt(end)

    assert not (earlier & later).any()
    assert not (later & test_mask).any()
    assert train.loc[earlier, 'had_claim'].nunique() == 2
    assert train.loc[later, 'had_claim'].nunique() == 2

    folds.append((start, earlier, later))
    fold_summary.append({
        'validation_start': start,
        'validation_end_exclusive': end,
        'fit_rows': int(earlier.sum()),
        'fit_claims': int(train.loc[earlier, 'had_claim'].sum()),
        'validation_rows': int(later.sum()),
        'validation_claims': int(train.loc[later, 'had_claim'].sum()),
    })
pd.DataFrame(fold_summary)

The 2022 cohort stays outside model selection and the model used to report its performance. Once evaluation is recorded, its labelled rows also contribute to the final scoring model.

## Explore risk patterns in development history

Use only 2020-2021 development history for outcome-based exploration. Compare fleet size, same-coverage prior claims and external risk score. Plot APD and AL separately so their different baseline rates do not hide other associations.

In [0]:
# Compare prior claims in the same coverage: APD history for APD, AL history for AL.
own_prior_claims = np.where(
    fit_data.coverage_type.eq('AL'),
    fit_data['prior_al_claim_count'],
    fit_data['prior_apd_claim_count']
)
plot_groups = [
    (pd.cut(fit_data.vehicle_count, [-0.1, 5, 10, np.inf],
            labels=['0-5', '6-10', '11+']), 'Fleet size'),
    (pd.Series(own_prior_claims, index=fit_data.index)
        .clip(upper=2)
        .map({0: '0', 1: '1', 2: '2+'}),
        'Prior claims in the same coverage'),
    (pd.cut(fit_data['risk_score_external'], [0, 25, 50, 75, 100], include_lowest=True,
            labels=['0-25', '25-50', '50-75', '75-100']), 'External risk score'),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (groups, title) in zip(axes, plot_groups):
    rates = fit_data.groupby([groups, 'coverage_type'], observed=True)['had_claim'].mean().unstack()
    rates.reindex(columns=['APD', 'AL']).mul(100).plot(marker='o', ax=ax)
    ax.set(title=title, xlabel='', ylabel='Observed claim rate (%)', ylim=(0, 30))
    ax.grid(alpha=0.2)
    ax.legend(frameon=False)

plt.tight_layout()
plt.show()
plt.close()

Fleet size, prior claims in the relevant coverage, and the external score show useful associations with claim occurrence.

## Build and compare complete prediction pipelines

The binary target makes logistic regression and a gradient boosting classifier appropriate main candidates. Logistic regression gives a regularized linear benchmark. Boosting can capture nonlinear effects and interactions in these tabular inputs.

The pipeline selects approved fields, optionally derives ratios/logs, and learns preparation only from its fit partition. One-hot models use median imputation, missing-value indicators and category encoding. Linear models also standardize numeric inputs. Native categorical boosting learns missing-value branches and treats category codes as labels, without imposing numeric ordering.

A Poisson model fits full-period claim counts and converts its expected count to occurrence probability. That conversion requires a Poisson conditional count distribution, an extra assumption the classifiers do not need. Equal 365-day terms remove variation in policy-time exposure. No class weighting or oversampling changes the observed prevalence. A small fixed candidate list compares regularization, tree size, engineered features, categorical handling, calibration and premium exclusion.

In [0]:
candidates = []
for engineered in [False, True]:
    for C in [0.05, 0.2, 1.0]:
        candidates.append({
            'name': f'logistic_{"derived" if engineered else "raw"}_C{C:g}',
            'family': 'logistic',
            'engineered': engineered,
            'C': C
        })

for leaves in [4, 8]:
    for iterations in [80, 160]:
        candidates.append({
            'name': f'boosting_raw_L{leaves}_T{iterations}',
            'family': 'boosting',
            'max_leaf_nodes': leaves,
            'max_iter': iterations
        })

for leaves in [4, 8]:
    candidates.append({
        'name': f'boosting_derived_L{leaves}_T160',
        'family': 'boosting',
        'engineered': True,
        'max_leaf_nodes': leaves,
        'max_iter': 160
    })

candidates += [
    {'name': 'boosting_raw_sigmoid', 'family': 'boosting', 'max_leaf_nodes': 8,
     'max_iter': 160, 'calibrate': True},
    {'name': 'logistic_derived_sigmoid', 'family': 'logistic', 'engineered': True,
     'C': 0.2, 'calibrate': True},
    {'name': 'logistic_derived_no_premium', 'family': 'logistic', 'engineered': True,
     'C': 0.2, 'include_premium': False},
]

# Poisson use the same outer dates and selection metric
for leaves in [4, 8]:
    for iterations in [80, 160]:
        candidates.append({
            'name': f'boosting_native_L{leaves}_T{iterations}',
            'family': 'boosting', 'native_categories': True,
            'max_leaf_nodes': leaves, 'max_iter': iterations,
        })
for alpha in [0.01, 0.1, 1.0]:
    candidates.append({
        'name': f'poisson_raw_alpha{alpha:g}', 'family': 'poisson', 'alpha': alpha,
    })

# Test premium removal for the main nonlinear model too
for native in [False, True]:
    candidates.append({
        'name': f'boosting_{"native" if native else "onehot"}_no_premium',
        'family': 'boosting', 'native_categories': native,
        'max_leaf_nodes': 8, 'max_iter': 160, 'include_premium': False,
    })
for leaves in [4, 8]:
    for iterations in [80, 160]:
        candidates.append({
            'name': f'boosting_native_derived_L{leaves}_T{iterations}',
            'family': 'boosting', 'native_categories': True, 'engineered': True,
            'max_leaf_nodes': leaves, 'max_iter': iterations,
        })
for iterations in [80, 160]:
    candidates.append({
        'name': f'boosting_native_L16_T{iterations}', 'family': 'boosting',
        'native_categories': True, 'max_leaf_nodes': 16, 'max_iter': iterations,
    })
print(f'{len(candidates)} candidate configurations')

In [0]:
# Ratios/logs are deterministic transformations; validation decides whether they help
example_features = UnderwritingFeatures(engineered=True).fit_transform(X_fit)
example_features[DERIVED].head().round(3)

Selection metric: choose the lowest row-weighted validation log loss, which scores the probability assigned to each observed outcome. Brier score also assesses probability quality; ROC AUC and average precision assess ranking. Lower log loss/Brier and higher AUC/AP are better. These losses combine calibration and discrimination, so we also inspect a calibration plot.

We preserve the observed class prevalence. Class weights or oversampling could distort probabilities unless corrected.

In [0]:
def fit_candidate(config, X, y, insured_ids, claim_counts):
    model = build_model(config)
    if config.get('calibrate', False):
        # Keep every company's rows together while learning the sigmoid calibrator
        splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
        group_splits = list(splitter.split(X, y, groups=insured_ids))

        for fit_ids, calibration_ids in group_splits:
            assert set(insured_ids.iloc[fit_ids]).isdisjoint(insured_ids.iloc[calibration_ids])

        model = CalibratedClassifierCV(
            model, method='sigmoid', cv=group_splits, ensemble=False, n_jobs=1)
        
    # Poisson uses counts in fitting; all models are evaluated against had_claim
    fit_target = claim_counts if config['family'] == 'poisson' else y
    
    return model.fit(X, fit_target)

Sigmoid calibration is optional. Its inner folds keep every company's rows together while creating out-of-fold training predictions, then refit the base pipeline on that window's fit history. Those inner folds are company validation rather than temporal deployment simulations. Outer validation always stays separate.

Two reference predictions use only fit labels: the overall claim rate and separate APD/AL claim rates. Any complex model must show useful improvement over these simple baselines.

In [0]:
validation_rows = []
with threadpool_limits(limits=2):
    for start, earlier, later in folds:
        earlier_data, later_data = train.loc[earlier], train.loc[later]
        X_earlier, y_earlier = earlier_data[FEATURES], earlier_data['had_claim']
        X_later, y_later = later_data[FEATURES], later_data['had_claim']
        coverage_rates = earlier_data.groupby('coverage_type')['had_claim'].mean()
        probabilities = {
            'overall_rate': np.full(len(later_data), y_earlier.mean()),
            'coverage_rates': later_data['coverage_type'].map(coverage_rates).to_numpy(),
        }
        for config in candidates:
            model = fit_candidate(
                config, X_earlier, y_earlier,
                earlier_data['insured_id'], earlier_data['claim_count'])
            # Class 1 means at least one claim; class 0 means no claim
            probabilities[config['name']] = model.predict_proba(X_later)[:, 1]
        for name, p in probabilities.items():
            validation_rows.append({
                'candidate': name,
                'window': start,
                'rows': len(y_later),
                **probability_metrics(y_later, p)
            })
        print(f'Completed validation window starting {start}')
validation_results = pd.DataFrame(validation_rows)

In [0]:
metric_columns = ['log_loss', 'brier_score', 'roc_auc', 'average_precision']
# Log loss is additive over rows, so weighting by window size gives the combined loss
# AUC/AP below are weighted window summaries
validation_summary = validation_results.groupby('candidate') \
    .apply(lambda group: pd.Series({
        metric: np.average(group[metric], weights=group.rows)
        for metric in metric_columns
    }), include_groups=False) \
    .sort_values('log_loss')

candidate_names = [config['name'] for config in candidates]
selected_name = validation_summary.loc[candidate_names, 'log_loss'].idxmin()
selected_config = next(config for config in candidates if config['name'] == selected_name)
print('Selected configuration:', selected_config)
print('Values below are row-weighted means of the two window metrics.')
validation_summary.round(4)

### Read the validation evidence

The table above selects the configuration with lowest row-weighted validation log loss. Compare its loss with both baselines and the simpler models. The difference between close candidates is uncertain with two windows.

In the verified run, logistic regression with derived inputs and `C=0.05` wins at validation log loss 0.357573. Raw-input boosting reaches 0.357707. This very small difference supports the simpler selected pipeline under this protocol.

### Check generalization to separate companies

Fit the selected configuration in three company-disjoint folds within 2020-2021 development history. Compare against fit-derived coverage rates. This diagnostic never uses the 2022 labels and does not select another model. It asks how the chosen model behaves for companies absent from each fit, whereas the main temporal cohort includes both recurring and new companies.

In [0]:
splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
group_rows = []
with threadpool_limits(limits=2):
    for fold, (fit_ids, check_ids) in enumerate(
            splitter.split(X_fit, y_fit, groups=fit_data['insured_id']), start=1):
        earlier, later = fit_data.iloc[fit_ids], fit_data.iloc[check_ids]
        assert set(earlier['insured_id']).isdisjoint(later['insured_id'])

        model = fit_candidate(selected_config, earlier[FEATURES], earlier['had_claim'],
                              earlier['insured_id'], earlier['claim_count'])
        rates = earlier.groupby('coverage_type')['had_claim'].mean()

        for name, p in {
            'selected_model': model.predict_proba(later[FEATURES])[:, 1],
            'coverage_rates': later['coverage_type'].map(rates).to_numpy(),
        }.items():
            group_rows.append({
                'fold': fold,
                'model': name,
                'rows': len(later),
                **probability_metrics(later['had_claim'], p)
            })

group_diagnostic = pd.DataFrame(group_rows)
group_diagnostic.groupby('model')[metric_columns].mean().round(4)

## Fit the model used for evaluation

Fit the selected configuration on 2020-2021 development history. A separate final_model will be fitted on all usable history after performance is recorded.

In [0]:
with threadpool_limits(limits=2):
    evaluation_model = fit_candidate(
        selected_config, X_fit, y_fit,
        fit_data['insured_id'], fit_data['claim_count'])
print(f'Fitted {selected_name} on {len(X_fit):,} policy lines '
      f'with {y_fit.sum():,} claim-positive outcomes')

## Inspect the fitted preparation

For one-hot candidates, the displayed medians come only from development history.

In [0]:
base_pipeline = (evaluation_model.calibrated_classifiers_[0].estimator
                 if selected_config.get('calibrate', False) else evaluation_model)
feature_frame = base_pipeline.named_steps['underwriting'].transform(X_fit)
numeric_columns = [col for col in feature_frame if col not in CATEGORICAL]
preprocessor = base_pipeline.named_steps['preprocess']
prepared_test = preprocessor.transform(
    base_pipeline.named_steps['underwriting'].transform(X_test))

if selected_config.get('native_categories', False):
    assert not np.isinf(prepared_test).any()
    preparation_summary = pd.DataFrame({
        'Missing fit values handled by boosting': feature_frame.isna().sum()
    })
else:
    imputer = preprocessor.named_transformers_['numeric'].named_steps['impute']
    preparation_summary = pd.DataFrame({
        'Missing fit values': feature_frame[numeric_columns].isna().sum(),
        'Fitted median': imputer.statistics_,
    })
    assert np.isfinite(prepared_test).all()
print('Prepared test shape:', prepared_test.shape, '| missing values:', np.isnan(prepared_test).sum())
preparation_summary.loc[preparation_summary.iloc[:, 0].gt(0)].round(3)

## Evaluate the 2022 historical cohort

Use evaluation_model for every metric below.

In [0]:
with threadpool_limits(limits=2):
    test_probability = evaluation_model.predict_proba(X_test)[:, 1]
fit_coverage_rates = fit_data.groupby('coverage_type')['had_claim'].mean()
test_baselines = {
    'overall_rate': np.full(len(y_test), y_fit.mean()),
    'coverage_rates': test_data['coverage_type'].map(fit_coverage_rates).to_numpy(),
    'selected_model': test_probability,
}
test_results = pd.DataFrame({
    name: dict(probability_metrics(y_test, p), mean_predicted_risk=float(np.mean(p)))
    for name, p in test_baselines.items()
}).T
print(f'Observed test claim rate: {y_test.mean():.2%}')
test_results.round(4)

A good risk score should both rank claim-prone policies and assign sensible probabilities. The precision-recall plot addresses ranking at different thresholds. The calibration plot compares the mean predicted probability with the observed claim rate in six equally sized risk groups.

In [0]:
precision, recall, _ = precision_recall_curve(y_test, test_probability)
calibration_data = pd.DataFrame({
    'predicted': test_probability,
    'observed': y_test.to_numpy()
})
# Equal-sized groups make each observed rate use roughly the same number of policies.
calibration_data['risk_group'] = pd.qcut(calibration_data.predicted, q=6, duplicates='drop')
calibration_groups = calibration_data.groupby('risk_group', observed=True).agg(
    predicted=('predicted', 'mean'), observed=('observed', 'mean'),
    policy_lines=('observed', 'size'), claims=('observed', 'sum'))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(recall, precision, label='Selected model')
axes[0].axhline(y_test.mean(), linestyle='--', color='grey', label='Test claim prevalence')
axes[0].set(title='Claim ranking on later policies', xlabel='Recall', ylabel='Precision')
axes[1].plot(calibration_groups['predicted'],
             calibration_groups['observed'],
             marker='o',
             label='Risk groups')

upper = max(calibration_groups['predicted'].max(), calibration_groups['observed'].max()) * 1.15
axes[1].plot([0, upper], [0, upper], linestyle='--', color='grey', label='Perfect agreement')
axes[1].set(title='Predicted versus observed risk', xlabel='Mean predicted probability',
            ylabel='Observed claim rate', xlim=(0, upper), ylim=(0, upper))

for ax in axes:
    ax.legend(frameon=False)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()
plt.close()
calibration_groups.round(4)

## Check coverage and company history

Evaluate APD and AL separately, then compare companies seen in the fit history with companies absent from it. These are diagnostics. A weak result in a small subgroup calls for more evidence.

In [0]:
seen_company = test_data['insured_id'].isin(set(fit_data['insured_id'])).to_numpy()
subgroup_masks = {
    'APD': test_data['coverage_type'].eq('APD').to_numpy(),
    'AL': test_data['coverage_type'].eq('AL').to_numpy(),
    'Company seen in fit': seen_company,
    'Company unseen in fit': ~seen_company,
}
subgroup_rows = []
for name, mask in subgroup_masks.items():
    labels, p = y_test.to_numpy()[mask], test_probability[mask]
    subgroup_rows.append(dict(
        subgroup=name, rows=int(mask.sum()), claims=int(labels.sum()),
        observed_rate=float(labels.mean()), predicted_rate=float(p.mean()),
        **probability_metrics(labels, p)))
subgroup_results = pd.DataFrame(subgroup_rows).set_index('subgroup')
subgroup_results.round(4)

## Quantify the gain over coverage-only scoring

Compute a paired difference in per-policy log loss: coverage baseline loss minus model loss. Positive values favor the model. Resample insured companies, keeping their policy rows together, to obtain a descriptive 95% bootstrap interval.

In [0]:
def per_policy_log_loss(labels, probabilities):
    p = np.clip(np.asarray(probabilities), 1e-12, 1 - 1e-12)
    return -(labels * np.log(p) + (1 - labels) * np.log1p(-p))


labels = y_test.to_numpy()
baseline_loss = per_policy_log_loss(labels, test_baselines['coverage_rates'])
model_loss = per_policy_log_loss(labels, test_probability)
paired = pd.DataFrame({'insured_id': test_data.insured_id.to_numpy(),
                       'gain': baseline_loss - model_loss})
clusters = paired.groupby('insured_id').gain.agg(['sum', 'size']).to_numpy()

rng = np.random.default_rng(SEED)
bootstrap_gains = []
for _ in range(1000):
    sample = clusters[rng.integers(0, len(clusters), size=len(clusters))]
    bootstrap_gains.append(sample[:, 0].sum() / sample[:, 1].sum())
gain_interval = np.quantile(bootstrap_gains, [0.025, 0.975])

print(f'Mean log-loss improvement: {paired.gain.mean():.4f}')
print(f'Company-bootstrap 95% interval: [{gain_interval[0]:.4f}, {gain_interval[1]:.4f}]')

if gain_interval[0] <= 0:
    print('The interval includes no gain; evidence of improvement is limited in this cohort.')
else:
    print('The interval favors the model in this cohort; future-period validation is still needed.')

## Inspect which inputs matter to prediction

Permutation importance shuffles one raw predictor at a time and measures the resulting increase in test log loss. Larger increases indicate stronger reliance on that input

In [0]:
with threadpool_limits(limits=2):
    # sklearn maximizes negative log loss. A positive importance is
    # a loss increase after shuffling
    importance = permutation_importance(
        evaluation_model, X_test, y_test, scoring='neg_log_loss', n_repeats=5,
        random_state=SEED, n_jobs=1)
importance_values = pd.Series(importance['importances_mean'], index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
importance_values.tail(8).plot.barh(ax=ax)
ax.set(title='Inputs the model relies on most',
       xlabel='Mean increase in log loss after shuffling',
       ylabel='')
ax.axvline(0, color='grey', linewidth=0.8)
ax.grid(axis='x', alpha=0.2)

plt.tight_layout()
plt.show()
plt.close()

In [0]:
loss_reduction = 1 - test_results.loc['selected_model', 'log_loss'] / test_results.loc['coverage_rates', 'log_loss']
# Selecting a fixed review capacity is different from declaring a probability threshold.
top_count = int(np.ceil(0.1 * len(y_test)))
top_positions = np.argsort(-test_probability, kind='stable')[:top_count]
top_rate = labels[top_positions].mean()
print(f'Test log-loss reduction versus coverage rates: {loss_reduction:.1%}')
print(f'Test ROC AUC: {test_results.loc["selected_model", "roc_auc"]:.3f}')
print(f'Test average precision: {test_results.loc["selected_model", "average_precision"]:.3f}; '
      f'claim prevalence: {y_test.mean():.3f}')
print(f'Top 10% of ranked policies: {top_rate:.1%} observed claims '
      f'({top_rate / y_test.mean():.2f} times the overall test rate)')

## Train the scoring model on all usable labelled history

The evaluation above is complete. Refit the same selected configuration, including its learned preparation, on all 14,990 retained historical rows.

In [0]:
with threadpool_limits(limits=2):
    final_model = fit_candidate(
        selected_config, train[FEATURES],
        train['had_claim'], train['insured_id'], train['claim_count'])
print(f'Final scoring model fitted on all {len(train):,} usable labelled rows')

# Provenance records the supplied cohort, not an invented historical availability cutoff
import hashlib
metadata = {
    'selected_config': selected_config,
    'training_snapshot_start': train['snapshot_date'].min().date().isoformat(),
    'training_snapshot_end': train['snapshot_date'].max().date().isoformat(),
    'fit_rows': len(train),
    'fit_claims': int(train['had_claim'].sum()),
    'policy_durations_days': sorted(
        (train.policy_expiration_date - train.policy_effective_date)
        .dt.days.unique().astype(int).tolist()
    ),
    'training_reference': training_reference(train),
    'training_quality': training_quality,
}
manifest = save_model(final_model, ARTIFACT_DIR, metadata)
loaded_model, loaded_manifest = load_model(ARTIFACT_DIR)
with threadpool_limits(limits=2):
    predictions, score_quality = score_policies(loaded_model, score_raw, loaded_manifest)
predictions.head()

In [0]:
# Verify both the output contract and parity between in-memory and reloaded models.
assert len(predictions) == len(score_raw)
assert predictions.policy_id.is_unique
assert predictions.policy_id.tolist() == score_raw.policy_id.tolist()
assert np.isfinite(predictions.risk_score).all()
assert predictions.risk_score.between(0, 1).all()
with threadpool_limits(limits=2):
    original_probability = final_model.predict_proba(score[FEATURES])[:, 1]
np.testing.assert_allclose(predictions.risk_score, original_probability, rtol=0, atol=1e-12)

predictions.to_csv(PROJECT_DIR / 'predictions.csv', index=False)
(REPORT_DIR / 'scoring_quality.json').write_text(json.dumps(score_quality, indent=2))
print(f'Saved {len(predictions):,} finite probabilities in original policy order')
print('New scoring category counts:', score_quality['unseen_categories'])

In [0]:
score_results = predictions.assign(coverage_type=score.coverage_type.to_numpy()) \
    .groupby('coverage_type')['risk_score'] \
    .agg(policy_lines='size',
         mean_risk='mean',
         median_risk='median',
         min_risk='min',
         max_risk='max')
score_results.round(4)

In [0]:
fig, ax = plt.subplots(figsize=(8.5, 3.8))
for coverage in ['APD', 'AL']:
    values = predictions.loc[score['coverage_type'].eq(coverage), 'risk_score']
    ax.hist(
        values,
        bins=np.linspace(0, max(0.4, np.ceil(predictions['risk_score'].max() * 10) / 10), 25),
        histtype='step', linewidth=2,
        weights=np.full(len(values), 100 / len(values)),
        label=coverage
    )

ax.set(title='Predicted claim risk in the scoring population',
       xlabel='Full-policy probability of at least one claim',
       ylabel='Policy lines within coverage (%)')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.show()
plt.close()

## Modeling methodology improvements

| Original prototype | Current implementation | Why the change matters |
| ------------------ | ---------------------- | ---------------------- |
| Current-period paid claims and reporting delay included in predictors | Fixed underwriting allowlist excludes current outcomes and current-year late payments | Removes information revealing the answer or unavailable at the intended scoring moment |
| Numeric missing values replaced with zero; independent categorical integer codes in train and scoring | Fitted pipeline with medians, missing indicators, scaling and shared category encoding | Unknown values remain distinct from real zeros, and category meanings remain consistent |
| Random split; accuracy computed on training rows | Expanding development windows, separate 2022 evaluation, simple baselines and probability/ranking metrics | Measures generalization to later supplied records and avoids training accuracy as evidence
| One default logistic fit and hard class predictions exported | Different configurations compared by log loss; probabilities exported with predict_proba | Selects a suitable model and produces an interpretable risk probability |
| Evaluation-fit model also used for final scoring | Fixed configuration refitted on all 14,990 usable labels after evaluation | Uses all valid supplied history for final predictions without confusing refit with evaluation

## Production readiness improvements

The original prototype reimplements preprocessing separately for each CSV, keeps the model only in notebook memory and writes predictions without an explicit output. The following changes address those concrete gaps. These are implemented prototype foundations.
1. One preparation and inference contract. Training and batch inference call the same prepare_data and score_policies functions. They validate required columns, IDs, coverage, dates, policy terms and output probabilities. Training exclusions are written to an audit file. Scoring rejects invalid identities or policy intervals rather than silently dropping policies. Fixed validity rules and learned model preparation have distinct responsibilities.
2. Persist the entire fitted prediction pipeline. save_model stores derived-feature construction, fitted imputation, encoding, scaling and the classifier together. The manifest records feature names, target, selected configuration, fit cohort, row count, seed, dependency versions, input/code hashes and the label interpretation. load_model checks schema and environment compatibility. Only trusted model artifacts should be loaded.
3. Verify the prediction output and serialization. The notebook checks 3,000 unique policy IDs in original input order, finite scores in [0, 1].
4. Retain reviewable run evidence and descriptive input monitoring. The next cell saves validation, evaluation, subgroup, calibration, company-bootstrap and importance results. The existing scoring report records missingness, unseen categories and numeric mean shifts against training statistics. It flags 60 new rideshare lines. These are saved diagnostics, not an automated alerting service or proof of accuracy on unlabelled scoring data.

In [0]:
# Persist evidence from evaluation_model
validation_results.to_csv(REPORT_DIR / 'validation_by_window.csv', index=False)
validation_summary.to_csv(REPORT_DIR / 'validation_summary.csv')
group_diagnostic.to_csv(REPORT_DIR / 'group_disjoint_diagnostic.csv', index=False)
test_results.to_csv(REPORT_DIR / 'test_metrics.csv')
subgroup_results.to_csv(REPORT_DIR / 'test_subgroups.csv')
calibration_groups.to_csv(REPORT_DIR / 'test_calibration.csv')
pd.DataFrame({
    'policy_id': test_data['policy_id'].to_numpy(),
    'insured_id': test_data['insured_id'].to_numpy(),
    'coverage_type': test_data['coverage_type'].to_numpy(),
    'had_claim': y_test.to_numpy(),
    'risk_score': test_probability,
}).to_csv(REPORT_DIR / 'test_predictions.csv', index=False)
pd.DataFrame({
    'feature': FEATURES,
    'mean_log_loss_increase': importance['importances_mean'],
    'repeat_std': importance['importances_std'],
}).to_csv(REPORT_DIR / 'permutation_importance.csv', index=False)

evaluation_report = {
    'selected_config': selected_config,
    'protocol': 'Retrospective capture-date cohorts, not an as-of deployment backtest',
    'evaluation_fit_rows': len(fit_data),
    'evaluation_rows': len(test_data),
    'evaluation_claims': int(y_test.sum()),
    'final_fit_rows': len(train),
    'metrics': test_results.to_dict(orient='index'),
    'log_loss_reduction_vs_coverage': float(loss_reduction),
    'paired_log_loss_gain': float(paired['gain'].mean()),
    'company_bootstrap_95pct_interval': gain_interval.tolist(),
    'top_decile': {
        'rows': top_count,
        'claims': int(labels[top_positions].sum()),
        'observed_rate': float(top_rate),
        'lift': float(top_rate / y_test.mean()),
    },
    'data_checks': data_checks,
    'validation_windows': fold_summary,
    'versions': manifest['versions'],
}
(REPORT_DIR / 'evaluation.json').write_text(
    json.dumps(evaluation_report, indent=2, allow_nan=False))
print('Saved development validation and 2022 evaluation evidence in reports/.')
print('All reported evaluation metrics belong to evaluation_model, fitted on 2020-2021.')

| Stage | Further work | Completion criterion |
| --- | --- | --- |
| 1. Package a reproducible batch workflow | Add CI checks for schema errors, unseen categories, missing values, and output row accounting. Store immutable data/model/run versions. Benchmark time and memory on expected volumes. | A repeatable run in the target environment, with versioned inputs/outputs and a demonstrated failure/recovery path. |
| 2. Run shadow scoring | Schedule the batch job without changing underwriting decisions. Log score versions, input quality and runtime. Check new rideshare business and shifted driver/mileage distributions. Compare observed outcomes once sufficiently mature. | Reliable scheduled runs and reviewed performance/quality evidence over the agreed observation period. |
| 3. Introduce a controlled rollout | Start with a limited decision-support use. Apply access controls, score audit trails, manual review and a rollback option. Assign incident ownership and service targets. | Approved scope, user guidance, traceable decisions and a tested rollback process. |
| 4. Monitor and maintain | Monitor missingness/category drift, runtime and scoring failures immediately. Monitor calibration, ranking, APD/AL and subgroup results when labels mature. Compare challengers, investigate alerts and retrain only with approved mature labels. | Defined monitoring cadence, thresholds, investigation actions, retraining/promotion rules and named owners. |